In [13]:
# =========================================================
# MULTIPLE LINEAR REGRESSION FROM SCRATCH
# USING MINI BATCH GRADIENT DESCENT (MBGD)
# =========================================================

import numpy as np

from sklearn.datasets import load_diabetes


class MLR_MBGD:

    # =====================================================
    # CONSTRUCTOR
    # =====================================================
    def __init__(
        self,
        learning_rate=0.01,
        epochs=1000,
        batch_size=32
    ):

        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size

        self.intercept_ = 0
        self.coef_ = None


    # =====================================================
    # TRAIN TEST SPLIT
    # =====================================================
    def train_test_split(
        self,
        X,
        y,
        test_size=0.2,
        random_state=None
    ):

        X = np.array(X)
        y = np.array(y)

        n = len(X)

        indices = np.arange(n)

        np.random.seed(random_state)
        np.random.shuffle(indices)

        X = X[indices]
        y = y[indices]

        split_index = int(n * (1 - test_size))

        X_train = X[:split_index]
        X_test = X[split_index:]

        y_train = y[:split_index]
        y_test = y[split_index:]

        return X_train, X_test, y_train, y_test


    # =====================================================
    # FIT MODEL USING MINI BATCH GD
    # =====================================================
    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y)

        n_samples = X.shape[0]
        n_features = X.shape[1]

        # Initialize Parameters
        self.coef_ = np.zeros(n_features)

        self.intercept_ = 0


        # =================================================
        # TRAINING LOOP
        # =================================================
        for epoch in range(self.epochs):

            # Shuffle Data Every Epoch
            indices = np.arange(n_samples)

            np.random.shuffle(indices)

            X = X[indices]
            y = y[indices]


            # =============================================
            # MINI BATCH LOOP
            # =============================================
            for i in range(0, n_samples, self.batch_size):

                X_batch = X[i:i + self.batch_size]

                y_batch = y[i:i + self.batch_size]

                batch_n = len(X_batch)

                # Predictions
                y_pred = (
                    np.dot(X_batch, self.coef_)
                    + self.intercept_
                )

                # Gradients
                d_coef = (
                    (-2 / batch_n)
                    * np.dot(X_batch.T, (y_batch - y_pred))
                )

                d_intercept = (
                    (-2 / batch_n)
                    * np.sum(y_batch - y_pred)
                )

                # Update Parameters
                self.coef_ = (
                    self.coef_
                    - self.learning_rate * d_coef
                )

                self.intercept_ = (
                    self.intercept_
                    - self.learning_rate * d_intercept
                )


    # =====================================================
    # PREDICTION
    # =====================================================
    def predict(self, X):

        X = np.array(X)

        y_pred = (
            np.dot(X, self.coef_)
            + self.intercept_
        )

        return y_pred


    # =====================================================
    # EVALUATION METRICS
    # =====================================================
    def evaluation(self, y_true, y_pred, X_test):

        y_true = np.array(y_true)
        y_pred = np.array(y_pred)

        # MSE
        mse = np.mean((y_true - y_pred) ** 2)

        # RMSE
        rmse = np.sqrt(mse)

        # MAE
        mae = np.mean(np.abs(y_true - y_pred))

        # R2 Score
        ss_total = np.sum(
            (y_true - np.mean(y_true)) ** 2
        )

        ss_residual = np.sum(
            (y_true - y_pred) ** 2
        )

        r2 = 1 - (ss_residual / ss_total)

        # Adjusted R2
        n = len(y_true)
        p = X_test.shape[1]

        adjusted_r2 = 1 - (
            ((1 - r2) * (n - 1))
            / (n - p - 1)
        )

        # Print Metrics
        print("MSE         :", mse)

        print("RMSE        :", rmse)

        print("MAE         :", mae)

        print("R2 Score    :", r2)

        print("Adjusted R2 :", adjusted_r2)


# =========================================================
# LOAD DIABETES DATASET
# =========================================================

diabetes = load_diabetes()

X = diabetes.data
y = diabetes.target


# =========================================================
# MODEL
# =========================================================

model = MLR_MBGD(
    learning_rate=0.01,
    epochs=1000,
    batch_size=32
)


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = model.train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =========================================================
# TRAIN MODEL
# =========================================================

model.fit(X_train, y_train)


# =========================================================
# PREDICTION
# =========================================================

y_pred = model.predict(X_test)


# =========================================================
# PARAMETERS
# =========================================================

print("Intercept :")
print(model.intercept_)

print()

print("Coefficients :")
print(model.coef_)


# =========================================================
# EVALUATION
# =========================================================

model.evaluation(y_test, y_pred, X_test)


# =========================================================
# IMPORTANT CONCEPTS
# =========================================================

# Model:
# Multiple Linear Regression

# Objective:
# OLS
#
# min ||y - Xb||²

# Optimization:
# MBGD (Mini Batch Gradient Descent)


# HOW MBGD WORKS
# ---------------------------------------------------------

# MBGD updates parameters after
# a mini batch of samples.


# MBGD UPDATE RULE
# ---------------------------------------------------------

# b = b - learning_rate * gradient


# WHY "MINI BATCH"?
# ---------------------------------------------------------

# Because updates happen using
# small batches instead of:
#
# - full dataset (BGD)
# - single sample (SGD)


# COMPARISON
# ---------------------------------------------------------

# BGD  -> Full dataset per update
# SGD  -> One sample per update
# MBGD -> Mini batch per update


# ADVANTAGES OF MBGD
# ---------------------------------------------------------

# 1. Faster than BGD
# 2. More stable than SGD
# 3. Efficient for large datasets
# 4. Better GPU utilization


# DISADVANTAGES
# ---------------------------------------------------------

# 1. Requires batch size tuning
# 2. Still approximate optimization

Intercept :
153.5512675287071

Coefficients :
[  56.635133    -45.37476435  288.14448417  181.53180615   23.2726498
   -6.81092329 -172.88990892  149.16111949  266.83743854  143.12770754]
MSE         : 3601.2195939135718
RMSE        : 60.01016242198959
MAE         : 50.47534546993815
R2 Score    : 0.3532008182952049
Adjusted R2 : 0.27027784628176954
